## Tổng kết đáp án
1C
2D
3B
4C
5C
6A
7C
8A
9A
10C

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data/'

def load(name):
  return pd.read_csv(DATA_DIR + name)

_cache = {}
def get(name):
  if name not in _cache: _cache[name] = load(name)
  return _cache[name].copy()

### Q1. Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

In [2]:
orders = get('orders.csv')
orders['order_date'] = pd.to_datetime(orders['order_date'])

orders = orders.sort_values(by=['customer_id', 'order_date'])
orders['prev_order_date'] = orders.groupby('customer_id')['order_date'].shift(1)
orders['inter_order_gap'] = (orders['order_date'] - orders['prev_order_date']).dt.days

gap = orders['inter_order_gap'].dropna().median()

print(f"Q1: Trung vị số ngày giữa hai lần mua liên tiếp là: {gap} ngày.")

for label, val in [('A) 30', 30), ('B) 90', 90), ('C) 144', 144), ('D) 365', 365)]:
    print(f'  {label} ngày  → diff = {abs(gap - val):.1f}')
print('ĐÁP ÁN C')

Q1: Trung vị số ngày giữa hai lần mua liên tiếp là: 144.0 ngày.
  A) 30 ngày  → diff = 114.0
  B) 90 ngày  → diff = 54.0
  C) 144 ngày  → diff = 0.0
  D) 365 ngày  → diff = 221.0
ĐÁP ÁN C


### Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?

In [3]:
products = get('products.csv')
products['gross_margin'] = (products['price'] - products['cogs'])/products['price']

result = products.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)
print(f"Q2: Phân khúc sản phẩm có tỷ suất lợi nhuận gộp trung bình cao nhất là: {result.idxmax()}")
print('ĐÁP ÁN D')

Q2: Phân khúc sản phẩm có tỷ suất lợi nhuận gộp trung bình cao nhất là: Standard
ĐÁP ÁN D


### Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

In [4]:
returns = get('returns.csv')
products = get('products.csv')

merged = returns.merge(products[['product_id', 'category']], on='product_id', how='left')
streetwear = merged[merged['category'] == 'Streetwear']

reason_counts = streetwear['return_reason'].value_counts()
print(f"Phổ biến nhất là {reason_counts.idxmax()} ({reason_counts.max():,} lần)")
print('ĐÁP ÁN B')

Phổ biến nhất là wrong_size (7,626 lần)
ĐÁP ÁN B


### Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

In [5]:
wt = get('web_traffic.csv')

result = wt.groupby('traffic_source')['bounce_rate'].mean().sort_values()
print(f"Thấp nhất là {result.idxmin()} ({result.min():.4f})")
print('ĐÁP ÁN C')

Thấp nhất là email_campaign (0.0045)
ĐÁP ÁN C


### Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?

In [6]:
items = get('order_items.csv')
total = len(items)
with_promo = items['promo_id'].notna().sum()
pct = with_promo/total * 100

print(f"Dòng có promo_id not null: {with_promo:,} / {total:,}")
print(f"Tỷ lệ: {pct:.2f}")
for label, val in [('A) 12%', 12), ('B) 25%', 25), ('C) 39%', 39), ('D) 54%', 54)]:
    print(f'  {label}  → diff = {abs(pct - val):.2f}%')
print('ĐÁP ÁN C')

Dòng có promo_id not null: 276,316 / 714,669
Tỷ lệ: 38.66
  A) 12%  → diff = 26.66%
  B) 25%  → diff = 13.66%
  C) 39%  → diff = 0.34%
  D) 54%  → diff = 15.34%
ĐÁP ÁN C


### Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)

In [7]:
customers = get('customers.csv')
orders = get('orders.csv')

cust_valid = customers[customers['age_group'].notna()][['customer_id','age_group']]

order_counts = orders.groupby('customer_id')['order_id'].count().reset_index()
order_counts.columns = ['customer_id', 'num_orders']

merged = cust_valid.merge(order_counts, on='customer_id', how='left')
merged['num_orders'] = merged['num_orders'].fillna(0)

result = merged.groupby('age_group').apply(
    lambda g: g['num_orders'].sum() / len(g)
).sort_values(ascending=False)

print(f"Cao nhất là {result.idxmax()} ({result.max():.4f})")
print('ĐÁP ÁN A')

Cao nhất là 55+ (5.4069)
ĐÁP ÁN A


### Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?

In [8]:
orders = get('orders.csv')
items = get('order_items.csv')
geo = get('geography.csv')

orders['order_date'] = pd.to_datetime(orders['order_date'])

start_date = pd.to_datetime('2017-07-04')
end_date = pd.to_datetime('2022-12-31')
train_orders = orders[(orders['order_date'] >= start_date) & (orders['order_date'] <= end_date)]

merged = pd.merge(items, train_orders, on='order_id', how='inner')
merged = pd.merge(merged, geo, on='zip', how='inner')
merged['line_revenue'] = merged['quantity'] * merged['unit_price']

revenue_by_region = merged.groupby('region')['line_revenue'].sum().sort_values(ascending=False)
highest_region = revenue_by_region.idxmax()
print(f"Tổng doanh thu theo từng vùng: {revenue_by_region}")
print(f"Vùng có doanh thu cao nhất: {highest_region}")
print("ĐÁP ÁN C")

Tổng doanh thu theo từng vùng: region
East       3.212665e+09
Central    2.244927e+09
West       1.529090e+09
Name: line_revenue, dtype: float64
Vùng có doanh thu cao nhất: East
ĐÁP ÁN C


### Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?

In [9]:
orders = get('orders.csv')
print(orders['order_status'].unique())

cancelled = orders[orders['order_status'] == 'cancelled']
result = cancelled['payment_method'].value_counts()

print(f"Phổ biến nhất là {result.idxmax()} với {result.max():,} đơn")
print('ĐÁP ÁN A')

<StringArray>
['delivered', 'returned', 'shipped', 'cancelled', 'paid', 'created']
Length: 6, dtype: str
Phổ biến nhất là credit_card với 28,452 đơn
ĐÁP ÁN A


### Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?

In [10]:
returns  = get('returns.csv')
items    = get('order_items.csv')
products = get('products.csv')

valid_sizes = ['S', 'M', 'L', 'XL']
prod_size = products[products['size'].isin(valid_sizes)][['product_id','size']]

items_with_size = items.merge(prod_size, on='product_id', how='inner')
denominator = items_with_size.groupby('size').size().rename('total_items')

returns_with_size = returns.merge(prod_size, on='product_id', how='inner')
numerator = returns_with_size.groupby('size').size().rename('total_returns')

combined = pd.concat([numerator, denominator], axis=1).reindex(valid_sizes)
combined['return_rate'] = combined['total_returns'] / combined['total_items']
combined = combined.sort_values('return_rate', ascending=False)

print(f'Cao nhất: {combined["return_rate"].idxmax()} ({combined["return_rate"].max():.4f})')
print('ĐÁP ÁN A')

Cao nhất: S (0.0565)
ĐÁP ÁN A


### Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?

In [11]:
payments = get('payments.csv')
result = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
print(f'Cao nhất: {result.idxmax()} kỳ ({result.max():,.2f})')
print('ĐÁP ÁN C')

Cao nhất: 6 kỳ (24,446.65)
ĐÁP ÁN C
